# THREE YEAR VARIANCE
Only accounting codes with data available for the previous three years (excluding COIVD years 2019–2022) will be included in this analysis.
A separate report can be generated for new or recently added account codes.Additionally, monthly variance may introduce outlier errors (e.g., large bills paid slightly late). Therefore, quarterly variance is considered more accurate and reliable for comparison.

In [1]:
import pandas as pd
df_26 = pd.read_excel('/kaggle/input/scotts/FY26.xlsx')
df_26 = df_26.fillna(0)
df_26.head(10)

,GL Account,DATE,GL Description,YTD Debits,YTD Credits
0,12345,2026-10-01,Hi World,20.00,0.0
1,12345,2026-10-01,Hi World,12.00,0.0
2,77777,2026-10-01,Hope,100.00,0.0
3,82800,2026-10-01,"Purpose, Romans",8.28,0.0
4,11111,2026-10-01,Winner,0.00,1.0
5,31600,2026-11-01,John,3.16,0.0
6,20260,2026-10-01,2026 ONLY EXTRA,5.00,0.0


In [2]:
# Read the Excel file into a DataFrame
df_24 = pd.read_excel('/kaggle/input/scotts/FY24.xlsx')
df_24 = df_24.fillna(0)

# Clean up column names by stripping any leading/trailing spaces
df_24.columns = df_24.columns.str.strip()

# Create a new column 'Year_24' to calculate the difference between 'YTD Debits' and 'YTD Credits'
df_24['FY24'] = df_24['YTD Debits'] - df_24['YTD Credits']

# Group by 'GL Account', summing the 'Year_24' values, and reset index to make it clean
df_24 = df_24.groupby(['GL Account', 'GL Description'], as_index=False)['FY24'].sum()
df_24.head()

,GL Account,GL Description,FY24
0,11111,Winner,-1.0
1,12345,Hi World,32.0
2,20240,2024 ONLY EXTRA,5.0
3,77777,Hope,100.0
4,82800,"Purpose, Rom",10.0


In [3]:
df_25 = pd.read_excel('/kaggle/input/scotts/FY25.xlsx')
df_25 = df_25.fillna(0)

# Strip leading/trailing spaces from column names
df_25.columns = df_25.columns.str.strip()

# Create the 'Ending-Amount' column
df_25['FY25'] = df_25['YTD Debits'] - df_25['YTD Credits']

# Group by 'Account ID' and sum the 'Balance' column
df_25 = df_25.groupby('GL Account')['FY25'].sum().reset_index()
df_25.head()

,GL Account,FY25
0,11111,-9.0
1,12345,32.0
2,20250,4.0
3,77777,100.0
4,82800,10.0


In [4]:
# Strip leading/trailing spaces from column names
df_26.columns = df_26.columns.str.strip()

# Create the 'Ending-Amount' column
df_26['FY26'] = df_26['YTD Debits'] - df_26['YTD Credits']

# Group by 'Account ID' and sum the 'Balance' column
df_26 = df_26.groupby('GL Account')['FY26'].sum().reset_index()
df_26.head()

,GL Account,FY26
0,11111,-1.00
1,12345,32.00
2,20260,5.00
3,31600,3.16
4,77777,100.00


# JOIN DATASETS

In [5]:
df= df_24.merge(df_25, on='GL Account', how='inner').merge(df_26, on='GL Account', how='inner')
df.head()

,GL Account,GL Description,FY24,FY25,FY26
0,11111,Winner,-1.0,-9.0,-1.00
1,12345,Hi World,32.0,32.0,32.00
2,77777,Hope,100.0,100.0,100.00
3,82800,"Purpose, Rom",10.0,10.0,8.28


# Outlier 
Using a two year baseline, we’ll treat FY24 and FY25 as the base years, then compare FY26 against the average of those two years, rounded to 2 decimals

In [6]:
outlier_percent = 10 # can change the outlier percentage as needed. 
print(outlier_percent, " outlier_percent")

# Step 1: Merge FY24 and FY25 (take the average as baseline):
df['Avg_FY24_FY25'] = df[['FY24', 'FY25']].mean(axis=1).round(2)

# Step 2: Calculate % deviation of FY26 from baseline:
df['Deviation_%'] = abs(df['FY26'] - df['Avg_FY24_FY25']) / abs(df['Avg_FY24_FY25']) * 100

# Step 3: Flag outliers (deviation > outlier_percent):
# Step 3: Replace Boolean with directional indicator
def classify_outlier(row):
    if row['Deviation_%'] > outlier_percent:
        if row['FY26'] > row['Avg_FY24_FY25']:
            return 'Increase'
        elif row['FY26'] < row['Avg_FY24_FY25']:
            return 'Decrease'
    return False  # Not an outlier

df['FY26_Outlier'] = df.apply(classify_outlier, axis=1)

# Show results
print(df[['GL Account', 'GL Description', 'FY24', 'FY25', 'Avg_FY24_FY25', 'FY26', 'Deviation_%', 'FY26_Outlier']])
df.head()

10  outlier_percent
   GL Account GL Description   FY24   FY25  Avg_FY24_FY25    FY26  \
0       11111         Winner   -1.0   -9.0           -5.0   -1.00   
1       12345       Hi World   32.0   32.0           32.0   32.00   
2       77777           Hope  100.0  100.0          100.0  100.00   
3       82800   Purpose, Rom   10.0   10.0           10.0    8.28   

   Deviation_% FY26_Outlier  
0         80.0     Increase  
1          0.0        False  
2          0.0        False  
3         17.2     Decrease  


,GL Account,GL Description,FY24,FY25,FY26,Avg_FY24_FY25,Deviation_%,FY26_Outlier
0,11111,Winner,-1.0,-9.0,-1.00,-5.0,80.0,Increase
1,12345,Hi World,32.0,32.0,32.00,32.0,0.0,False
2,77777,Hope,100.0,100.0,100.00,100.0,0.0,False
3,82800,"Purpose, Rom",10.0,10.0,8.28,10.0,17.2,Decrease
